# Tutorial 4 - Analysis

We can now do the exciting part of any simulation analysis! As has often been said, what you do now will heavily depend on what you are interested in about your system. The analysis below is inspired by analysis currently being performed for active projects, such as looking at lipid association and membrane thinning around a protein. We will also look at the basics such as root mean square deviation (RMSD) and root mean square fluctuation (RMSF) of the protein to get an idea of what is happening in the simulation. 

If there is time, we can also look at some more in-depth analysis tools for picking out specifics about lipid binding (which can easily be used for ligand binding) and conversion to atomistic resolution from a Martini coordinate file. 

In [ ]:
!gmx trjconv -h

First, we need to make sure our trajectories are processed to keep the protein central, which will make sure there are no large spikes when measuring properties such as RMSD and RMSF. We can do that using the `gmx trjconv` command, which converts trajectories (as the name suggests). The flags we will use are:

- `-f` the input of either a coordinate file (`.gro`/`.pdb`) or trajectory file (`.xtc`) to be altered
- `-s` the input `.tpr` file we used for our simulation; this contains information about the different molecules present etc
- `-o` the output name for our new file, depending on what the input was this will either a coordinate file (`.gro`/`.pdb`) or trajectory file (`.xtc`)
- `-center` which specifies the box should be redefined around a molecule in the center. This is the first of the selections, and we select the protein with `1`
- `-pbc` which defines how we define the periodic boundary treatment. `mol` means that the center of mass of the molecule will be placed inside of the box, and should prevent long bonds from being drawn during visualisation
- `-ur` sets the unit cell representation and is used with `-pbc`. In our case, we had a rectangular/square box, so we will define it as `rect`

The last `0` for each command selects the whole system to output. When large systems are simulated, it could be useful to remove parts of the system that are not needed for analysis.

We also make a new index file to select just the backbone beads for some of our analysis.

In [ ]:
%%bash
cd tutorial_4

gmx trjconv -f md.gro -s md.tpr -o md.pdb -center -pbc mol -ur rect << EOF
1
0
EOF

gmx trjconv -f md.xtc -s md.tpr -o md-center.xtc -center -pbc mol -ur rect << EOF
1
0
EOF

gmx make_ndx -f md.gro -o system.ndx << EOF
a BB
q
EOF

rm \#*
ls

Now is a good time to look at our trajectory to make sure everything looks fine. 

In [ ]:
from IPython.display import HTML, display
import base64, os, json, tempfile
import numpy as np                    

# ── NEW: optional MDAnalysis for binary trajectory formats ──────────────────
try:
    import MDAnalysis as mda
    _HAS_MDA = True
except ImportError:
    _HAS_MDA = False


# ══════════════════════════════════════════════════════════════════════════════
# ── Built-in residue-name sets
# ══════════════════════════════════════════════════════════════════════════════

_PROTEIN_RESNAMES: set = {
    # Standard amino acids (GROMACS / AMBER / CHARMM conventions)
    'ALA', 'ARG', 'ASN', 'ASP', 'CYS', 'GLN', 'GLU', 'GLY',
    'HIS', 'HIE', 'HID', 'HIP', 'HSE', 'HSD', 'HSP',
    'ILE', 'LEU', 'LYS', 'MET', 'PHE', 'PRO', 'SER', 'THR',
    'TRP', 'TYR', 'VAL',
    # Terminal caps
    'ACE', 'NME', 'NHE', 'NH2', 'CT3',
}

_WATER_RESNAMES: set = {
    'W',    # Martini 3 standard water (4-to-1 mapping)
    'WF',   # Martini 3 antifreeze water
}


# ══════════════════════════════════════════════════════════════════════════════
# ── Static-mode helpers  (unchanged from v2)
# ══════════════════════════════════════════════════════════════════════════════

def _split_by_resnames(text: str, fmt: str, resnames: set) -> tuple:
    """
    Partition a PDB or GRO structure string into (matched_text, unmatched_text).

    Non-atom records (REMARK, CONECT, END, GRO box vector) are duplicated into
    both outputs so each sub-file is self-consistent and parseable by Mol*.
    """
    if not resnames:
        return '', text

    if fmt == 'pdb':
        matched, unmatched = [], []
        for line in text.splitlines(keepends=True):
            if line[:6] in ('ATOM  ', 'HETATM'):
                (matched if line[17:21].strip() in resnames else unmatched).append(line)
            else:
                matched.append(line)
                unmatched.append(line)
        return ''.join(matched), ''.join(unmatched)

    elif fmt == 'gro':
        lines = text.splitlines(keepends=True)
        if len(lines) < 3:
            return '', text
        title = lines[0]
        try:
            n = int(lines[1].strip())
        except ValueError:
            return '', text
        box = lines[2 + n] if len(lines) > 2 + n else '\n'

        matched, unmatched = [], []
        for line in lines[2:2 + n]:
            rn = line[5:10].strip() if len(line) >= 10 else ''
            (matched if rn in resnames else unmatched).append(line)

        m_text = title + f'{len(matched)}\n'   + ''.join(matched)   + box
        u_text = title + f'{len(unmatched)}\n' + ''.join(unmatched) + box
        return m_text, u_text

    else:
        print(f"[view_martini] Warning: component splitting requires PDB or GRO "
              f"(got '{fmt}'). Loading full structure as a single layer.")
        return text, ''


def _has_atoms(text: str, fmt: str) -> bool:
    """Return True if the structure text contains at least one atom record."""
    if fmt == 'pdb':
        return any(ln[:6] in ('ATOM  ', 'HETATM') for ln in text.splitlines())
    if fmt == 'gro':
        lines = text.splitlines()
        try:
            return len(lines) >= 2 and int(lines[1].strip()) > 0
        except ValueError:
            return False
    return bool(text.strip())


def _color_theme(color: str) -> dict:
    """
    Convert a color string to a Mol* colorTheme descriptor dict.

    CSS hex  →  uniform theme with value encoded as a 24-bit int.
    Other str → Mol* named theme (e.g. 'chain-id', 'residue-name').
    """
    if isinstance(color, str) and color.startswith('#'):
        return {'name': 'uniform', 'params': {'value': int(color.lstrip('#'), 16)}}
    return {'name': color}


# ══════════════════════════════════════════════════════════════════════════════
# ── NEW ── Trajectory helpers
# ══════════════════════════════════════════════════════════════════════════════

def _write_multimodel_pdb(ag, stride: int = 1, start: int = 0, stop=None) -> str:
    """
    Write frames of an MDAnalysis AtomGroup to a multi-model PDB string.

    MDAnalysis's PDB writer requires an on-disk path, so a temporary file is
    used and immediately read back.  The result is a string of MODEL … ENDMDL
    blocks (one per frame) that Mol* recognises natively as a trajectory.

    Parameters
    ----------
    ag     : MDAnalysis.AtomGroup   Atoms to write (may be a selection subset).
    stride : int                    Write every Nth frame (default 1 → all).
    start  : int                    First frame index (default 0).
    stop   : int or None            Exclusive end-frame index (None → all).

    Returns
    -------
    str  Multi-model PDB content.
    """
    u          = ag.universe
    stop_frame = stop if stop is not None else u.trajectory.n_frames

    with tempfile.NamedTemporaryFile(suffix='.pdb', delete=False) as tmp:
        tmpname = tmp.name
    try:
        with mda.Writer(tmpname, ag.n_atoms, multiframe=True, reindex=False) as W:
            for _ in u.trajectory[start:stop_frame:stride]:
                W.write(ag)
        with open(tmpname, 'r') as f:
            return f.read()
    finally:
        if os.path.exists(tmpname):
            os.unlink(tmpname)


def _trajectory_component(
    u,
    resnames: set,
    used_indices: set,
    stride: int,
    start: int,
    stop,
) -> tuple:
    """
    Select atoms by residue name (excluding already-used atoms) and produce
    a multi-model PDB string covering the requested frame range.

    Implements sequential-peeling so no bead appears in more than one component
    layer, mirroring the behaviour of _split_by_resnames in static mode.

    Parameters
    ----------
    u            : MDAnalysis.Universe
    resnames     : set of str   Residue names to include.
    used_indices : set of int   Atom indices already claimed by prior components.
    stride / start / stop       Forwarded to _write_multimodel_pdb.

    Returns
    -------
    (pdb_text, updated_used_indices) : (str, set)
        pdb_text is '' when no atoms match.
    """
    if not resnames:
        return '', used_indices

    sel_str = 'resname ' + ' '.join(sorted(resnames))
    ag = u.select_atoms(sel_str)

    if len(ag) == 0:
        return '', used_indices

    # Exclude atoms already assigned to a prior component
    if used_indices:
        mask = ~np.isin(ag.indices, list(used_indices))
        ag   = ag[mask]

    if len(ag) == 0:
        return '', used_indices

    pdb_text     = _write_multimodel_pdb(ag, stride, start, stop)
    used_indices = used_indices | set(ag.indices.tolist())
    return pdb_text, used_indices


# ══════════════════════════════════════════════════════════════════════════════
# ── HTML / iframe builder  (updated for trajectory flag)
# ══════════════════════════════════════════════════════════════════════════════

def _build_html(
    components: list,
    height: int,
    show_animation: bool = False,   # ── NEW parameter
) -> HTML:
    """
    Assemble the Mol* iframe page from a list of component descriptor dicts.

    Each dict must contain:
        b64           : str   base64-encoded structure / trajectory text
        fmt           : str   Mol* format string ('pdb', 'gro', 'mmcif')
        label         : str   display label (shown in Mol* state tree)
        spacefillSize : float bead sphere radius in Å
        alpha         : float sphere opacity (0–1)
        showBonds     : bool  whether to render bond sticks
        stickRadius   : float bond stick radius in Å
        colorTheme    : dict  Mol* colorTheme descriptor

    show_animation : bool                                           ── NEW ──
        When True, sets viewportShowAnimation: true in Mol*, revealing the
        frame-stepper and play button in the viewport toolbar.
    """
    comps_json   = json.dumps(components)
    # Python bool → JS bool literal
    show_anim_js = 'true' if show_animation else 'false'

    # Note: Python f-string — all JS curly braces doubled {{ }}
    #       except Python variables being interpolated.
    page = f"""<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8"/>
  <link rel="stylesheet"
        href="https://cdn.jsdelivr.net/npm/molstar@latest/build/viewer/molstar.css"/>
  <style>
    html, body {{ margin:0; padding:0; height:100%; background:#1a1a2e; }}
    #app {{ position:absolute; inset:0; }}
  </style>
</head>
<body>
  <div id="app"></div>
  <script src="https://cdn.jsdelivr.net/npm/molstar@latest/build/viewer/molstar.js"></script>
  <script>
  (async () => {{

    // ── 1. Create Mol* viewer ─────────────────────────────────────────────────
    // isTrajectory is injected from Python; it gates the animation UI.
    const isTrajectory = {show_anim_js};

    const viewer = await molstar.Viewer.create('app', {{
      layoutIsExpanded:      false,
      layoutShowControls:    false,
      layoutShowLeftPanel:   false,
      layoutShowSequence:    false,
      layoutShowLog:         false,
      viewportShowAnimation: isTrajectory,  // ← NEW: shows frame slider + ▶ button
    }});
    const plugin = viewer.plugin;

    // ── 2. Component descriptors injected from Python ─────────────────────────
    const components = {comps_json};

    // ── 3. Helper: load one component and apply representations ───────────────
    async function loadComponent(comp) {{
      const rawData = await plugin.builders.data.rawData(
        {{ data: atob(comp.b64) }},
        {{ state: {{ isGhost: true }} }}
      );

      // parseTrajectory handles both single-model PDB and multi-model PDB;
      // Mol* automatically counts MODEL blocks and creates a frame sequence.
      const traj   = await plugin.builders.structure.parseTrajectory(rawData, comp.fmt);
      const model  = await plugin.builders.structure.createModel(traj);
      const struct = await plugin.builders.structure.createStructure(model,
                       {{ label: comp.label }});

      // VdW spacefill — one sphere per CG bead
      await plugin.builders.structure.representation.addRepresentation(struct, {{
        type:       'spacefill',
        colorTheme: comp.colorTheme,
        sizeTheme:  {{ name: 'uniform', params: {{ value: comp.spacefillSize }} }},
        typeParams: {{ alpha: comp.alpha }},
      }});

      // Bond sticks (optional per component)
      if (comp.showBonds) {{
        await plugin.builders.structure.representation.addRepresentation(struct, {{
          type:       'ball-and-stick',
          colorTheme: comp.colorTheme,
          sizeTheme:  {{ name: 'uniform' }},
          typeParams: {{
            sizeFactor:      comp.stickRadius,
            sizeAspectRatio: 1,
            alpha:           comp.alpha,
          }},
        }});
      }}

      return traj;
    }}

    // ── 4. Load all components in sequence ────────────────────────────────────
    const trajectories = [];
    for (const comp of components) {{
      const traj = await loadComponent(comp);
      if (traj) trajectories.push(traj);
    }}

    // ── 5. Focus camera on the full scene ────────────────────────────────────
    plugin.managers.camera.reset();

    // ── 6. NEW: Trajectory animation setup ───────────────────────────────────
    // viewportShowAnimation: true (set above) already exposes the frame slider
    // and ▶ play button in the Mol* viewport toolbar.
    //
    // When the user clicks ▶, Mol*'s built-in AnimateModelIndex advances ALL
    // loaded structures in lock-step — so protein, lipids, water, and ions
    // all move together even though they are separate Mol* state nodes.
    //
    // We pre-register the animation here (then immediately pause it) so the
    // fps / loop settings are applied when the user hits play.
    if (isTrajectory && trajectories.length > 0) {{
      try {{
        const animKey = 'animate-model-index';
        const registry = plugin.state.animation && plugin.state.animation.registry;
        if (registry && registry.has(animKey)) {{
          await plugin.managers.animation.play(
            registry.get(animKey),
            {{
              duration: {{ name: 'computed', params: {{ targetFps: 5 }} }},
              mode:     {{ name: 'loop',     params: {{}} }},
            }}
          );
          // Start paused — user clicks ▶ when ready
          // plugin.managers.animation.stop();
        }}
      }} catch (e) {{
        // Gracefully degrade if animation API differs across Mol* versions.
        // The frame slider in the toolbar still works regardless.
        console.warn('[view_martini] Animation pre-registration skipped:', e.message);
      }}
    }}

  }})();
  </script>
</body>
</html>"""

    encoded = base64.b64encode(page.encode()).decode()
    return HTML(
        f'<iframe src="data:text/html;base64,{encoded}" '
        f'width="100%" height="{height}px" style="border:none;"></iframe>'
    )


# ══════════════════════════════════════════════════════════════════════════════
# ── Main public function
# ══════════════════════════════════════════════════════════════════════════════

def view_martini(
    structure_file: str,
    # ── NEW: Trajectory parameters ────────────────────────────────────────────
    trajectory_file: str  = None,
    stride: int           = 1,
    start_frame: int      = 0,
    end_frame: int        = None,
    # ── Display ───────────────────────────────────────────────────────────────
    height: int = 600,
    # ── Protein ───────────────────────────────────────────────────────────────
    show_protein: bool          = True,
    protein_names: list         = None,
    protein_color: str          = '#4A90D9',
    protein_bead_radius: float  = 2.6,
    protein_alpha: float        = 0.55,
    show_protein_bonds: bool    = True,
    protein_stick_radius: float = 0.25,
    # ── Lipids ────────────────────────────────────────────────────────────────
    show_lipids: bool          = True,
    lipid_names: list          = None,
    lipid_color: str           = '#E8A838',
    lipid_bead_radius: float   = 2.6,
    lipid_alpha: float         = 0.55,
    show_lipid_bonds: bool     = True,
    lipid_stick_radius: float  = 0.25,
    # ── Water ─────────────────────────────────────────────────────────────────
    show_water: bool          = False,
    water_names: list         = None,
    water_color: str          = '#44AAFF',
    water_bead_radius: float  = 2.3,
    water_alpha: float        = 0.25,
    # ── Ions ──────────────────────────────────────────────────────────────────
    show_ions: bool          = True,
    ion_names: list          = None,
    ion_color: str           = '#FF4444',
    ion_bead_radius: float   = 2.1,
    ion_alpha: float         = 0.85,
) -> HTML:
    """
    Visualise a Martini 3 CG structure or trajectory in Mol* inside Jupyter.

    Two modes
    ---------
    Static (trajectory_file=None):
        Reads structure_file directly; no extra dependencies.
        Supports PDB, GRO, and mmCIF/CIF.

    Trajectory (trajectory_file provided):
        Requires MDAnalysis (pip install MDAnalysis).
        Uses structure_file as topology and trajectory_file for coordinates.
        Per-component multi-model PDB files are written to a temp directory
        and streamed to Mol*, which animates all layers in lock-step.

    Parameters
    ----------
    structure_file : str
        Path to a GRO or PDB file.  In trajectory mode this is the topology;
        in static mode it is the sole input.

    trajectory_file : str, optional                                 ── NEW ──
        Path to a binary trajectory or a multi-model PDB.
        Supported formats via MDAnalysis:
            XTC / TRR  — GROMACS compressed / full-precision trajectories
            DCD        — NAMD / CHARMM trajectory format
            multi-model PDB — alternatively, pass directly as structure_file
                              (MDAnalysis not required in that case).
        Raises ImportError if MDAnalysis is not installed.

    stride : int, optional                                          ── NEW ──
        Load every Nth frame (default 1 → all frames).
        Increase for faster loading of long simulations.

    start_frame : int, optional                                     ── NEW ──
        First frame index to load, 0-based (default 0).

    end_frame : int or None, optional                               ── NEW ──
        Exclusive last-frame index (None → up to final frame).
        Example: start_frame=100, end_frame=500, stride=5
                 → loads frames 100, 105, 110, …, 495.

    height : int
        Viewer height in pixels (default 600).

    show_protein / show_lipids / show_water / show_ions : bool
        Toggle each component layer on or off.

    protein_names / lipid_names / water_names / ion_names : list of str
        Residue names for each component.
        protein uses a built-in amino-acid set; water defaults to {'W','WF'}.
        lipid_names and ion_names must be given explicitly, e.g.:
            lipid_names=['POPC', 'POPE', 'CHOL']
            ion_names  =['NA', 'CL', 'CA']

    *_color : str
        CSS hex colour ('#RRGGBB') or a Mol* named theme
        ('chain-id', 'residue-name', …).

    *_bead_radius : float
        Spacefill sphere radius in Å:
            Tiny (T) beads  → 2.1 Å
            Small (S) beads → 2.3 Å
            Regular (R)     → 2.6 Å  (default)

    *_alpha : float
        Sphere opacity 0–1.

    show_protein_bonds / show_lipid_bonds : bool
        Overlay bond-stick representation on protein / lipid beads.

    *_stick_radius : float
        Bond stick radius in Å (default 0.25).

    Returns
    -------
    IPython.display.HTML
        Call display() on the return value or let Jupyter auto-display it.

    Examples
    --------
    Static view::

        display(view_martini('system.gro',
                             lipid_names=['POPC', 'POPE'],
                             ion_names=['NA', 'CL']))

    Full trajectory, every 10th frame::

        display(view_martini('system.gro',
                             trajectory_file='traj.xtc',
                             stride=10,
                             lipid_names=['POPC', 'POPE'],
                             ion_names=['NA', 'CL']))

    Production window only, water hidden::

        display(view_martini('system.gro',
                             trajectory_file='traj.xtc',
                             start_frame=500, end_frame=2000, stride=5,
                             lipid_names=['POPC'],
                             show_water=False))
    """

    # ── Resolve residue-name sets ─────────────────────────────────────────────
    protein_resnames = (set(protein_names) if protein_names else set()) | _PROTEIN_RESNAMES
    lipid_resnames   = set(lipid_names)  if lipid_names   else set()
    water_resnames   = set(water_names)  if water_names   else _WATER_RESNAMES
    ion_resnames     = set(ion_names)    if ion_names     else set()

    # ── Shared component-descriptor factory ───────────────────────────────────
    def _make_comp(b64, fmt, label, radius, alpha, show_bonds, stick_r, color):
        return {
            'b64':           b64,
            'fmt':           fmt,
            'label':         label,
            'spacefillSize': radius,
            'alpha':         alpha,
            'showBonds':     show_bonds,
            'stickRadius':   stick_r,
            'colorTheme':    _color_theme(color),
        }

    # ══════════════════════════════════════════════════════════════════════════
    # PATH A ── NEW: Trajectory mode  (MDAnalysis required)
    # ══════════════════════════════════════════════════════════════════════════
    if trajectory_file is not None:
        if not _HAS_MDA:
            raise ImportError(
                "MDAnalysis is required for binary trajectory loading.\n"
                "Install it with:  pip install MDAnalysis\n\n"
                "Tip: to avoid this dependency you can pre-convert your trajectory\n"
                "to a multi-model PDB (e.g. with gmx trjconv or MDAnalysis offline)\n"
                "and pass it directly as structure_file."
            )

        print(f"[view_martini] Opening topology:   {structure_file}")
        print(f"[view_martini] Opening trajectory: {trajectory_file}")
        u = mda.Universe(structure_file, trajectory_file)
        n_total  = u.trajectory.n_frames
        n_loaded = len(range(start_frame, end_frame or n_total, stride))
        print(f"[view_martini] {n_total} frames total → loading {n_loaded} "
              f"(start={start_frame}, stop={end_frame or n_total}, stride={stride})")

        components = []
        used       = set()  # atom indices already claimed

        # Sequential peeling: protein → lipids → water → ions
        if show_protein and protein_resnames:
            txt, used = _trajectory_component(u, protein_resnames, used,
                                              stride, start_frame, end_frame)
            if txt:
                b64 = base64.b64encode(txt.encode()).decode()
                components.append(_make_comp(b64, 'pdb', 'Protein (CG)',
                                             protein_bead_radius, protein_alpha,
                                             show_protein_bonds, protein_stick_radius,
                                             protein_color))
                print(f"[view_martini]   ✓ Protein — {n_loaded} frames written")

        if show_lipids and lipid_resnames:
            txt, used = _trajectory_component(u, lipid_resnames, used,
                                              stride, start_frame, end_frame)
            if txt:
                b64 = base64.b64encode(txt.encode()).decode()
                components.append(_make_comp(b64, 'pdb', 'Lipids (CG)',
                                             lipid_bead_radius, lipid_alpha,
                                             show_lipid_bonds, lipid_stick_radius,
                                             lipid_color))
                print(f"[view_martini]   ✓ Lipids  — {n_loaded} frames written")

        if show_water and water_resnames:
            txt, used = _trajectory_component(u, water_resnames, used,
                                              stride, start_frame, end_frame)
            if txt:
                b64 = base64.b64encode(txt.encode()).decode()
                components.append(_make_comp(b64, 'pdb', 'Water (CG)',
                                             water_bead_radius, water_alpha,
                                             False, 0.0, water_color))
                print(f"[view_martini]   ✓ Water   — {n_loaded} frames written")

        if show_ions and ion_resnames:
            txt, used = _trajectory_component(u, ion_resnames, used,
                                              stride, start_frame, end_frame)
            if txt:
                b64 = base64.b64encode(txt.encode()).decode()
                components.append(_make_comp(b64, 'pdb', 'Ions (CG)',
                                             ion_bead_radius, ion_alpha,
                                             False, 0.0, ion_color))
                print(f"[view_martini]   ✓ Ions    — {n_loaded} frames written")

        if not components:
            print("[view_martini] Warning: no atoms matched any component. "
                  "Check residue names.")

        return _build_html(components, height, show_animation=True)

    # ══════════════════════════════════════════════════════════════════════════
    # PATH B ── Static mode  (single structure file, no extra deps)
    # ══════════════════════════════════════════════════════════════════════════
    ext = os.path.splitext(structure_file)[1].lower().lstrip('.')
    fmt = {'pdb': 'pdb', 'gro': 'gro',
           'cif': 'mmcif', 'mmcif': 'mmcif'}.get(ext, 'pdb')

    with open(structure_file, 'r', encoding='utf-8', errors='replace') as fh:
        full_text = fh.read()

    components = []
    remaining  = full_text  # whittled down by sequential peeling

    if show_protein and protein_resnames:
        matched, remaining = _split_by_resnames(remaining, fmt, protein_resnames)
        if matched and _has_atoms(matched, fmt):
            b64 = base64.b64encode(matched.encode()).decode()
            components.append(_make_comp(b64, fmt, 'Protein (CG)',
                                         protein_bead_radius, protein_alpha,
                                         show_protein_bonds, protein_stick_radius,
                                         protein_color))

    if show_lipids and lipid_resnames:
        matched, remaining = _split_by_resnames(remaining, fmt, lipid_resnames)
        if matched and _has_atoms(matched, fmt):
            b64 = base64.b64encode(matched.encode()).decode()
            components.append(_make_comp(b64, fmt, 'Lipids (CG)',
                                         lipid_bead_radius, lipid_alpha,
                                         show_lipid_bonds, lipid_stick_radius,
                                         lipid_color))

    if show_water and water_resnames:
        matched, remaining = _split_by_resnames(remaining, fmt, water_resnames)
        if matched and _has_atoms(matched, fmt):
            b64 = base64.b64encode(matched.encode()).decode()
            components.append(_make_comp(b64, fmt, 'Water (CG)',
                                         water_bead_radius, water_alpha,
                                         False, 0.0, water_color))

    if show_ions and ion_resnames:
        matched, remaining = _split_by_resnames(remaining, fmt, ion_resnames)
        if matched and _has_atoms(matched, fmt):
            b64 = base64.b64encode(matched.encode()).decode()
            components.append(_make_comp(b64, fmt, 'Ions (CG)',
                                         ion_bead_radius, ion_alpha,
                                         False, 0.0, ion_color))

    if not components:
        print("[view_martini] Warning: no atoms matched any component. "
              "Loading full structure as a single layer.")
        b64 = base64.b64encode(full_text.encode()).decode()
        components.append(_make_comp(b64, fmt, 'Full Structure',
                                     2.6, 0.55, True, 0.25, '#4A90D9'))

    return _build_html(components, height, show_animation=False)



display(view_martini("md.pdb", trajectory_file='md-center.xtc',lipid_names=['POPC', 'POPE', 'POPI']))



**To get the animation to play, click the little gold rectangle with a play button in, which you can find in the top left corner, and then click start**

Now, we can start to actually to perform the commands for the analysis. Gromacs has commands built in to help with some of this, such as RMSD and RMSF analysis. For RMSD, we can use the `rms` command and select the backbone to align to, and the backbone to measure. For RMSF, we will use the `rmsf` command and select the backbone.

In [ ]:
%%bash 

cd tutorial_4

gmx rms -f md-center.xtc -s md.tpr -n system.ndx -o rmsd.xvg << EOF
18
18
EOF

gmx rmsf -f md-center.xtc -s md.tpr -n system.ndx -o rmsf.xvg << EOF
18
EOF

Now we have the data, but we need a way to visulise this! Below is a simple plotting script to look at how the RMSD of the protein changes over the course of the simulation:

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import rc, rcParams
import pylab
import csv
import os

mpl.rc_file_defaults()
plt.rcParams.update({'font.size':15})
fig,ax = plt.subplots(1, 1, figsize=(6,6))


def plot_RMSD(path_to_folder,colour):
	time   = np.genfromtxt('{}/rmsd.xvg'.format(path_to_folder), skip_header=18, usecols=(0)) / 1000000
	rmsd  =  np.genfromtxt('{}/rmsd.xvg'.format(path_to_folder), skip_header=18, usecols=(1)) * 10
	ax.plot(time,rmsd, color=colour,linewidth=2)
	maxtime = time.max()
	return maxtime

maxtime = plot_RMSD('./','gray')

ax.set_xlim(xmin=0, xmax=maxtime)
ax.set_xlabel("Time (μs)")
ax.set_ylabel("RMSD (Å)")


There is a lot of data here, so we can simplify the plotting by plotting a runing average over ten data points:

In [ ]:
mpl.rc_file_defaults()
plt.rcParams.update({'font.size':15})
fig,ax = plt.subplots(1, 1, figsize=(6,6))

def window(size):
    return np.ones(size)/float(size)

def plot_RMSD_smooth(path_to_folder,colour):
	time   = np.genfromtxt('{}/rmsd.xvg'.format(path_to_folder), skip_header=18, usecols=(0)) / 1000000
	rmsd  =  np.genfromtxt('{}/rmsd.xvg'.format(path_to_folder), skip_header=18, usecols=(1)) * 10
	ax.plot(time,rmsd, color=colour,linewidth=2, alpha=0.2)
	ax.plot(time,np.convolve(rmsd,window(10),'same'), color=colour,linewidth=2)
	maxtime = time.max()
	return maxtime

maxtime = plot_RMSD_smooth('./','gray')

ax.set_xlim(xmin=0, xmax=maxtime)
ax.set_xlabel("Time (μs)")
ax.set_ylabel("RMSD (Å)")


You can see here that the RMSD over the simulation is very small, due to the elastic network we implemented during the `martinize` step. The RMSD tells us about the global behaviour of the protein, but we can look at the movement of the individual residues using RMSF. We can plot that below:

In [ ]:
fig,ax = plt.subplots(1, 1, figsize=(6,6))

def plot_RMSF(path_to_folder,colour):
	res   =  np.genfromtxt('{}/rmsf.xvg'.format(path_to_folder), skip_header=18, usecols=(0))
	res   =  np.arange(len(res))
	rmsf  =  np.genfromtxt('{}/rmsf.xvg'.format(path_to_folder), skip_header=18, usecols=(1)) * 10
	ax.plot(res,rmsf, color=colour,linewidth=2)
	max_res = len(res)
	return max_res

max_res = plot_RMSF('./','gray')

ax.set_xlim(xmin=0,xmax=max_res)
ax.set_xlabel("Residue")
ax.set_ylabel("RMSF (Å)")


From this graph, you can see that even though there are elastic networks, some parts of the protein are more flexible than others. Thinking about the protein structure, which parts of the protein do you think would be most flexible?

<details>
<summary>    
Click here for the answer
</summary>

The loops of the protein that connect the β-sheets will be the most flexible, as they will have a limited elastic network holding them in place

</details>


If considering whether the force-constant and cutoffs used match behaviour seen in atomistic simulations, comparing RMSF between CG and AT simulations will show if the correct parts of your protein are being constrained or are flexible!

Do this **out of the notebook** in your own terminal:

```$vmd your_CG_system.pdb```

Now that we have had a look at how the protein behaves, we can look at the lipids around the proteins and how they interact. To do this, we are going to re-use some analysis already developed for generating lipid density plots around a fitted (not rotating) protein, originally used [here](https://www.nature.com/articles/s42003-025-08381-5). From this, we can see if there are any hotspots that might indicate binding sites

First, we need to load in the trajectories (this can take multiple simulations at the same time if we had them) and align to a reference plane to keep the protein fixed for making these plots. The trajectory reader (taw) also has a built-in timing function so we can see how long each step takes:

In [ ]:
import taw
import math
import glob

timing = taw.Timer()

with timing('Reading trajectories'):
    trjs = [
            taw.TrajectoryWithPBC(
                tpr=f'md.tpr', 
                trj=f'md.xtc', 
                selection='not resname W',
                step=1
            ).compact('protein')
    ]

with timing('Aligning all frames to common reference in xy plane'):
    reference = trjs[0]['protein'][-1].coords
    reference -= reference.mean(axis=0)
    trjs = [ trj.alignxy('protein', reference) for trj in trjs ]

print(timing)

The next step is separating the different components in the system and centring everything relative to the protein. We can also split our lipids into different species so we can look at each in detail:

In [ ]:
with timing('Selecting lipids'):
    lipids = [ trj['not protein and not resname ION'] for trj in trjs ]


with timing('Selecting ions'):
    ions = [ trj['resname ION'] for trj in trjs ]
    
with timing('Selecting protein'):
    protein = [ trj['protein'] for trj in trjs ]

with timing('Getting backbone'):
    backbone = [ trj['name BB'] for trj in trjs ]


with timing('Centering everything on mean membrane z'):
    for lip, prot, ion in zip(lipids, backbone, ions):
        mid = lip.coords[:, :, 2].mean(axis=1)
        lip[:, :, 2] -= mid[:, None]
        prot[:, :, 2] -= mid[:, None]
        ion[:, :, 2] -= mid[:, None]
        
with timing('Splitting lipids in single species'):
    lipidspecies = {
        species: [ t[f'resname {species}'] for t in lipids ]
        for species in set(lipids[0].universe.atoms.resnames)
    }

with timing('Splitting ions in single species'):
    ionspecies = {
        species: [ t[f'name {species}'] for t in ions ]
        for species in set(ions[0].universe.atoms.names)
    }

colormaps = { 'POPI':'RdPu', 'POPE':'Oranges','POPC':'Purples', 'NA':'Reds', 'CL':'Greens' }

print(timing)

Now to plot our densities! This next cell will plot a face-down view of the lipid density for each species found around the protein, which we show as sticks. The darker the colour for the lipid density, the more that lipid is found there

In [ ]:
with timing('Plot all'):
    for lip, lipco in lipidspecies.items():
        print('######## {} ########'.format(lip))
        plt.subplots(figsize=(6, 6))
        array_op,xedges,yedges = np.histogram2d(*lipco[0].coords[:, :, :2].reshape((-1, 2)).T, bins=100)
        max_array=max(array_op.flatten())
        array_avg = array_op/max_array
        extent = [xedges[0], xedges[-1], yedges[0], yedges[-1] ]
        plt.imshow(array_avg.T, interpolation='bicubic',extent=extent,origin='lower', cmap=colormaps[lip])
        plt.plot(*backbone[0].coords[:, :, :2].mean(axis=0).reshape((-1, 2)).T, c='gray')
        plt.gca().set_aspect('equal')
        plt.ylim(-50, 50)
        plt.xticks(np.arange(-50, 51, step=25))
        plt.xlim(-50, 50)
        plt.yticks(np.arange(-50, 51, step=25))
        plt.show()
        
print(timing)

Are there any lipid species you think accumulate? Where do you think that this is happening? 

<details>
<summary>    
Click here for the answer
</summary>

POPI looks to be accumulating in three positions around the protein, with the most defined density at the very top of the protein as it is plotted here. This could mean there are binding sites, which we will investigate further below 

</details>

We can also do the same for the side on view of the protein and bilayer:

In [ ]:
with timing('Plot all'):
    for lip, lipco in lipidspecies.items():
        print('######## {} ########'.format(lip))
        plt.subplots(figsize=(6, 6))
        array_op,xedges,yedges = np.histogram2d(*lipco[0]['name PO*'].coords[:, :, 1:3].reshape((-1, 2)).T, bins=50)
        max_array=max(array_op.flatten())
        array_avg = array_op/max_array
        extent = [xedges[0], xedges[-1], yedges[0], yedges[-1] ]
        plt.imshow(array_avg.T, interpolation='bicubic',extent=extent,origin='lower', cmap=colormaps[lip])
        plt.plot(*backbone[0].coords[:, :, 1:3].mean(axis=0).reshape((-1, 2)).T, c='gray')
        plt.gca().set_aspect('equal')
        plt.show()
        
print(timing)

Why do you think there are density differences between the different leaflets? 

<details>
<summary>    
Click here for the answer
</summary>

There is 6x more POPI on the lower leaflet and twice as much POPE on the upper leaflet as on the lower leaflet, causing the differences seen here. The amount of POPC is very similar between the two leaflets, so we do not observe a difference. 

</details>

As well as looking at the densities for lipid species, we can also look at the behaviour of ions in the system. _Caution should be taken not to overinterpret ion behaviour in simulations such as this, as their parameterisation is less well verified_. We can plot this in the same way as we did for the lipids:

In [ ]:
with timing('Plot all'):
    for ion, ionco in ionspecies.items():
        print('######## {} ########'.format(ion))
        plt.subplots(figsize=(6, 6))
        array_op,xedges,yedges = np.histogram2d(*ionco[0].coords[:, :, 1:3].reshape((-1, 2)).T, bins=50)
        max_array=max(array_op.flatten())
        array_avg = array_op/max_array
        extent = [xedges[0], xedges[-1], yedges[0], yedges[-1] ]
        plt.imshow(array_avg.T, interpolation='bicubic',extent=extent,origin='lower', cmap=colormaps[ion])
        plt.plot(*backbone[0].coords[:, :, 1:3].mean(axis=0).reshape((-1, 2)).T, c='gray')
        plt.gca().set_aspect('equal')
        plt.show()
        
print(timing)

We can still see species-specific behaviour, and this can be understood by looking at specific residues found inside the channel.

## Needs to be finished (mol* maybe)

As well as looking at specific accumulation, it is also interesting to look at the overall effects on how the protein affects membrane behaviour. This could include traits such as lipid ordering, but we are going to look at bilayer thickness. 

We will first create a map of the phosphate positions and then use this to work out how thick the bilayer is at certain positions. The average thickness for a phospholipid bilayer is ~4 Å, so we can use this as a sanity check

In [ ]:
# ── Spatially resolved thickness map — 2D binning ────────────────────────
#
# For each xy bin we collect all PO.* z-coordinates across all frames,
# split by leaflet (z > 0 = upper, z < 0 = lower), and compute
# thickness = mean(upper z) - mean(lower z) in that bin.

with timing('Spatial thickness map'):
    rep_trj = lipids[0]
    po_trj  = rep_trj['name PO*']

    # Flatten all frames: collect (x, y, z) for every PO bead across all frames
    coords_all = po_trj.coords.reshape(-1, 3)   # (n_frames * n_beads, 3)
    x_all = coords_all[:, 0]
    y_all = coords_all[:, 1]
    z_all = coords_all[:, 2]

    # 2D bin edges — cover the range of PO bead positions
    bin_width = 1.5 # Å — increase for smoother map, decrease for more detail
    x_edges = np.arange(x_all.min(), x_all.max() + bin_width, bin_width)
    y_edges = np.arange(y_all.min(), y_all.max() + bin_width, bin_width)
    nx, ny  = len(x_edges) - 1, len(y_edges) - 1

    # Bin indices for every bead-frame observation
    ix = np.searchsorted(x_edges[1:], x_all)   # which x bin
    iy = np.searchsorted(y_edges[1:], y_all)   # which y bin

    upper_sum = np.zeros((nx, ny))
    upper_cnt = np.zeros((nx, ny))
    lower_sum = np.zeros((nx, ny))
    lower_cnt = np.zeros((nx, ny))

    upper_mask = z_all > 0
    lower_mask = z_all < 0

    np.add.at(upper_sum, (ix[upper_mask], iy[upper_mask]), z_all[upper_mask])
    np.add.at(upper_cnt, (ix[upper_mask], iy[upper_mask]), 1)
    np.add.at(lower_sum, (ix[lower_mask], iy[lower_mask]), z_all[lower_mask])
    np.add.at(lower_cnt, (ix[lower_mask], iy[lower_mask]), 1)

    with np.errstate(invalid='ignore'):
        upper_z = np.where(upper_cnt > 0, upper_sum / upper_cnt, np.nan)
        lower_z = np.where(lower_cnt > 0, lower_sum / lower_cnt, np.nan)

    thickness_map = upper_z - lower_z   # (nx, ny), in Å

    # Bin centres for plotting
    x_centres = 0.5 * (x_edges[:-1] + x_edges[1:])
    y_centres = 0.5 * (y_edges[:-1] + y_edges[1:])

    valid = np.isfinite(thickness_map)
    print(f'Thickness — mean: {np.nanmean(thickness_map):.1f} Å, '
          f'min: {np.nanmin(thickness_map):.1f} Å, '
          f'max: {np.nanmax(thickness_map):.1f} Å')

print(timing)

We are seeing variation around 4 Å, but the mean seems sensible. Let's plot this relative to the protein to see what effect it is having. We can highlight a slightly unusual charged glutamine residue that sticks out into the bilayer with a red circle

In [ ]:
# ── Plot spatial thickness map ────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(6, 6))

valid = np.isfinite(thickness_map)
print('vmin  =' + str(np.percentile(thickness_map[valid], 2)))
print('vmax  =' + str(np.percentile(thickness_map[valid], 98)))

im = ax.imshow(
    thickness_map.T,
    origin='lower',
    extent=[x_edges[0], x_edges[-1], y_edges[0], y_edges[-1]],
    cmap='RdBu_r', vmin=32, vmax=48, # These limits will capture nearly all of the data and centers around the value we could expect for a phospholipid bilayer
    interpolation='bilinear',   # smooth pixel interpolation
    aspect='equal'
)
cbar = fig.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('Thickness (Å)')

ax.plot(*backbone[0].coords[:, :, :2].mean(axis=0).reshape((-1, 2)).T, c='gray')
ax.plot(*backbone[0].coords[:, :, :2].mean(axis=0).reshape((-1, 2))[72], c='red', marker='o')
ax.set_xlim(-50, 50)
ax.set_ylim(-50, 50)
ax.set_xticks(np.arange(-50, 51, step=25))
ax.set_yticks(np.arange(-50, 51, step=25))
ax.set_aspect('equal')
ax.set_xlabel('x (Å)')
ax.set_ylabel('y (Å)')
plt.show()

How would you describe the effect that our protein is having? What do you think the glutamine residue could be contributing to this? 

<details>
<summary>    
Click here for the answer
</summary>

The protein is thinning the membrane in most places, but mostly around where the positively charged residue points into the membrane core. This residue has been implicated in interactions with partner proteins, and thinning the membrane at this position is important to their association. You can read more about this [here](https://www.nature.com/articles/s42003-025-07551-9).

</details>

Now, we can revisit lipid interaction. While a density plot shows us that there could be some interaction points with POPI, the above analysis does not give us any specifics about the interaction. Various packages can help with this, including [ProLint2](https://github.com/ProLint/prolint2), but today we are going to look at [PyLipID](https://pylipid.readthedocs.io/en/master/).

This is a package specifically designed to look at protein-lipid interactions, at both CG and AT resolution. It can also take multiple simulations to get even more sampling and independent starting positions (i.e. if insane is used separately to set up each repeat, a new lipid arrangement will be simulated). It will output quite a few different findings in a folder called `Interaction_POPI`

The below script has been taken and modified from the [example script found on the PyLipID page](https://pylipid.readthedocs.io/en/master/demo.html). This can take a little while to run

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pylipid.api import LipidInteraction
from pylipid.util import check_dir

##################################################################
##### This part needs changes according to your setting ##########
##################################################################
trajfile_list = ["md.xtc"]
topfile_list = ["md.gro"]  # topology file is needed when trajectory format does not
                                               # provide topology information. See mdtraj.load() for more
                                               # information.
dt_traj = None  # the timestep of trajectories. Need to use this param when trajectories are in a format
                # with no timestep information. Not necessary for trajectory formats of e.g. xtc, trr.
stride = 10   # tell pylipid to analyze every stride-th frame. Can be used to save computation memory
             # and speed up the calculation.

lipid = "POPI"   # residue name in the topology.
lipid_atoms = None  # all lipid atoms will be considered for interaction calculation. Sometimes it is good to just consider the headgroup beads
cutoffs = [0.5, 0.7]  # dual-cutoff scheme for coarse-grained simulations. Single-cutoff scheme can be
                      # achieved by using the same value for two cutoffs.

nprot = 1   # if the simulation system has N copies of receptors, "nprot=N" will report interactions
            # averaged from the N copies, but "nprot=1" will ask PyLipID to report interaction for
            # each copy.

binding_site_size = 4  # binding site should contain at least four residues.

n_top_poses = 3     # write out num. of representative bound poses for each binding site.
n_clusters = "auto"  # cluster the bound poses for a binding site into num. of clusters. PyLipID
                     # will write out a pose conformation for each of the cluster. By default, i.e.
                     # "auto", PyLipID will use a density based clusterer to find possible clusters.

save_dir = None  # save at current working directory if it is None.
save_pose_format = "gro"  # format that poses are written in
save_pose_traj = False  # save all the bound poses in a trajectory for each binding site. The generated
                       # trajectories can take some disk space (up to a couple GB depending on your system).
save_pose_traj_format = "xtc"  # The format for the saved pose trajectories. Can take any format that is supported
                               # by mdtraj.

timeunit = "us"  # micro-sec. "ns" is nanosecond. Time unit used for reporting the results.
resi_offset = 0  # shift the residue index, useful for MARTINI models.

radii = None  # Radii of protein atoms/beads. In the format of python dictionary {atom_name: radius}
              # Used for calculation of binding site surface area. The van der waals radii of common atoms were
              # defined by mdtraj (https://github.com/mdtraj/mdtraj/blob/master/mdtraj/geometry/sasa.py#L56).
              # The radii of MARTINI 2.2 beads were included in PyLipID.

pdb_file_to_map = None   # if a pdb coordinate of the receptor is provided, a python script
                         # "show_binding_site_info.py" will be generated which maps the binding
                         # site information to the structure in PyMol. As PyMol cannot recognize
                         # coarse-grained structures, an atomistic structure of the receptor is needed.

fig_format = "png"  # format for all pylipid produced figures. Allow for formats that are supported by
                    # matplotlib.pyplot.savefig().

num_cpus = None  # the number of cpu to use when functions are using multiprocessing. By default,
                 # i.e. None, the functions will use up all the cpus available. This can use up all the memory in
                 # some cases.

#####################################
###### no changes needed below ######
#####################################

#### calculate lipid interactions
li = LipidInteraction(trajfile_list, topfile_list=topfile_list, cutoffs=cutoffs, lipid=lipid,
                      lipid_atoms=lipid_atoms, nprot=1, resi_offset=resi_offset,
                      timeunit=timeunit, save_dir=save_dir, stride=stride, dt_traj=dt_traj)
li.collect_residue_contacts()
li.compute_residue_duration(residue_id=None)
li.compute_residue_occupancy(residue_id=None)
li.compute_residue_lipidcount(residue_id=None)
li.show_stats_per_traj(write_log=True, print_log=True)
li.compute_residue_koff(residue_id=None, plot_data=True, fig_close=True,
                        fig_format=fig_format, num_cpus=num_cpus)
li.compute_binding_nodes(threshold=binding_site_size, print_data=False)
if len(li.node_list) == 0:
    print("*"*50)
    print("No binding site detected! Skip analysis for binding sites.")
    print("*"*50)
else:
    li.compute_site_duration(binding_site_id=None)
    li.compute_site_occupancy(binding_site_id=None)
    li.compute_site_lipidcount(binding_site_id=None)
    li.compute_site_koff(binding_site_id=None, plot_data=True, fig_close=True,
                         fig_format=fig_format, num_cpus=num_cpus)
    pose_traj, pose_rmsd_data = li.analyze_bound_poses(binding_site_id=None, pose_format=save_pose_format,
                                                       n_top_poses=n_top_poses, n_clusters=n_clusters,
                                                       fig_format=fig_format, num_cpus=num_cpus)
    # save pose trajectories
    if save_pose_traj:
        for bs_id in pose_traj.keys():
            pose_traj[bs_id].save("{}/Bound_Poses_{}/Pose_traj_BSid{}.{}".format(li.save_dir, li.lipid, bs_id,
                                                                          save_pose_traj_format))
    del pose_traj  # save memory space
    surface_area_data = li.compute_surface_area(binding_site_id=None, radii=radii, fig_format=fig_format)
    data_dir = check_dir(li.save_dir, "Dataset_{}".format(li.lipid))
    #pose_rmsd_data.to_csv("{}/Pose_RMSD_data.csv".format(data_dir), index=False, header=True)
    #surface_area_data.to_csv("{}/Surface_Area_data.csv".format(data_dir), index=True, header=True)
    li.write_site_info(sort_residue="Residence Time")

if pdb_file_to_map is not None:
    li.save_pymol_script(pdb_file_to_map)

#### write and save data
# for item in ["Dataset", "Duration", "Occupancy", "Lipid Count", "CorrCoef"]:
#     li.save_data(item=item)
# for item in ["Residence Time", "Duration", "Occupancy", "Lipid Count"]:
#     li.save_coordinate(item=item)
for item in ["Residence Time", "Duration", "Occupancy", "Lipid Count"]:
    li.plot(item=item, fig_close=True, fig_format=fig_format)
    li.plot_logo(item=item, fig_close=True, fig_format=fig_format)

#### plot binding site comparison.
if len(li.node_list) > 0:
    for item in ["Duration BS", "Occupancy BS"]:
        li.save_data(item=item)

        ylabel_timeunit = 'ns' if li.timeunit == "ns" else r"$\mu$s"
        ylabel_dict = {"Residence Time": "Residence Time ({})".format(ylabel_timeunit),
                       "Duration": "Duration ({})".format(ylabel_timeunit),
                       "Occupancy": "Occuoancy (100%)",
                       "Lipid Count": "Lipid Count (num.)"}

        # plot No. 1
        binding_site_IDs = np.sort(
                 [int(bs_id) for bs_id in li.dataset["Binding Site ID"].unique() if bs_id != -1])
        for item in ["Residence Time", "Duration", "Occupancy", "Lipid Count"]:
            item_values = np.array(
                      [li.dataset[li.dataset["Binding Site ID"]==bs_id]["Binding Site {}".format(item)].unique()[0]
                       for bs_id in binding_site_IDs])
            fig, ax = plt.subplots(1, 1, figsize=(len(li.node_list)*0.5, 2.6))
            ax.scatter(np.arange(len(item_values)), np.sort(item_values)[::-1], s=50, color="red")
            ax.set_xticks(np.arange(len(item_values)))
            sorted_index = np.argsort(item_values)[::-1]
            ax.set_xticklabels(binding_site_IDs[sorted_index])
            ax.set_xlabel("Binding Site ID", fontsize=12)
            ax.set_ylabel(ylabel_dict[item], fontsize=12)
            for label in ax.xaxis.get_ticklabels()+ax.yaxis.get_ticklabels():
                plt.setp(label, fontsize=12, weight="normal")
            plt.tight_layout()
            plt.show()
            #plt.savefig("{}/{}_{}_v_binding_site.{}".format(li.save_dir, li.lipid, "_".join(item.split()), fig_format),dpi=200)
            
        # plot No. 2
        binding_site_IDs_RMSD = np.sort([int(bs_id) for bs_id in binding_site_IDs
                                        if f"Binding Site {bs_id}" in pose_rmsd_data.columns])
        RMSD_averages = np.array(
                     [pose_rmsd_data[f"Binding Site {bs_id}"].dropna(inplace=False).mean()
                      for bs_id in binding_site_IDs_RMSD])
        fig, ax = plt.subplots(1, 1, figsize=(len(li.node_list)*0.5, 2.6))
        ax.scatter(np.arange(len(RMSD_averages)), np.sort(RMSD_averages)[::-1], s=50, color="red")
        ax.set_xticks(np.arange(len(RMSD_averages)))
        sorted_index = np.argsort(RMSD_averages)[::-1]
        ax.set_xticklabels(binding_site_IDs_RMSD[sorted_index])
        ax.set_xlabel("Binding Site ID", fontsize=12)
        ax.set_ylabel("RMSD (nm)", fontsize=12)
        for label in ax.xaxis.get_ticklabels()+ax.yaxis.get_ticklabels():
            plt.setp(label, fontsize=12, weight="normal")
        plt.tight_layout()
        plt.show()
        #plt.savefig("{}/{}_RMSD_v_binding_site.{}".format(li.save_dir, li.lipid, fig_format), dpi=200)

        # plot No. 3
        surface_area_averages = np.array(
                       [surface_area_data["Binding Site {}".format(bs_id)].dropna(inplace=False).mean()
                        for bs_id in binding_site_IDs])
        fig, ax = plt.subplots(1, 1, figsize=(len(li.node_list)*0.5, 2.6))
        ax.scatter(np.arange(len(surface_area_averages)), np.sort(surface_area_averages)[::-1], s=50, color="red")
        ax.set_xticks(np.arange(len(surface_area_averages)))
        sorted_index = np.argsort(surface_area_averages)[::-1]
        ax.set_xticklabels(binding_site_IDs[sorted_index])
        ax.set_xlabel("Binding Site ID", fontsize=12)
        ax.set_ylabel(r"Surface Area (nm$^2$)", fontsize=12)
        for label in ax.xaxis.get_ticklabels()+ax.yaxis.get_ticklabels():
            plt.setp(label, fontsize=12, weight="normal")
        plt.tight_layout()
        plt.show()
        #plt.savefig("{}/{}_surface_area_v_binding_site.{}".format(li.save_dir, li.lipid, fig_format), dpi=200)

        # plot No. 4
        res_time_BS = np.array(
                  [li.dataset[li.dataset["Binding Site ID"]==bs_id]["Binding Site Residence Time"].unique()[0]
                   for bs_id in binding_site_IDs_RMSD])
        fig, ax = plt.subplots(1, 1, figsize=(len(li.node_list)*0.5, 2.6))
        ax.scatter(res_time_BS, RMSD_averages, s=50, color="red")
        ax.set_xlabel(ylabel_dict["Residence Time"], fontsize=12)
        ax.set_ylabel("RMSD (nm)", fontsize=12)
        for label in ax.xaxis.get_ticklabels()+ax.yaxis.get_ticklabels():
            plt.setp(label, fontsize=12, weight="normal")
        plt.tight_layout()
        plt.show()
        #plt.savefig("{}/{}_Residence_Time_v_RMSD.{}".format(li.save_dir, li.lipid, fig_format), dpi=200)

        # plot No. 5
        res_time_BS = np.array(
                  [li.dataset[li.dataset["Binding Site ID"]==bs_id]["Binding Site Residence Time"].unique()[0]
                   for bs_id in binding_site_IDs])
        fig, ax = plt.subplots(1, 1, figsize=(len(li.node_list)*0.5, 2.6))
        ax.scatter(res_time_BS, surface_area_averages, s=50, color="red")
        ax.set_xlabel(ylabel_dict["Residence Time"], fontsize=12)
        ax.set_ylabel(r"Surface Area (nm$^2$)", fontsize=12)
        for label in ax.xaxis.get_ticklabels()+ax.yaxis.get_ticklabels():
            plt.setp(label, fontsize=12, weight="normal")
        plt.tight_layout()
        plt.show()
        #plt.savefig("{}/{}_Residence_Time_v_surface_area.{}".format(li.save_dir, li.lipid, fig_format), dpi=200)